# Paimon PROXY (JDBC-oriented) via Spark notebook

Registers a PAIMON PROXY catalog using JDBC-oriented catalog properties.
If this combination is unsupported by your runtime, switch `type` to `filesystem` in the payload.

In [ ]:
import json
import os
import urllib.request

base_url = os.environ.get("KASANARI_BASE_URL", "http://kasanari:9090")
jdbc_uri = os.environ.get("KASANARI_JDBC_URI", "jdbc:postgresql://catalog-storage:5432/postgres")
s3_endpoint = os.environ.get("KASANARI_S3_ENDPOINT", "http://minio:9000")
catalog_id = "paimon_spark_proxy_jdbc"

payload = {
    "catalogId": catalog_id,
    "catalogType": "PAIMON",
    "mode": "PROXY",
    "spec": {
        "fileIoProperties": {
            "fs.s3a.access.key": "admin",
            "fs.s3a.secret.key": "password",
            "fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
            "fs.s3a.path.style.access": "true",
            "fs.s3a.endpoint": s3_endpoint,
        },
        "catalogProperties": {
            "type": "jdbc",
            "warehouse": "s3a://warehouse",
            "jdbc-url": jdbc_uri,
            "jdbc-user": "postgres",
            "jdbc-password": "postgres",
            "jdbc-driver": "org.postgresql.Driver",
            "jdbc-table-prefix": "paimon_",
        },
    },
}

req = urllib.request.Request(
    f"{base_url}/management/v1/catalogs",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req) as resp:
    print("register status:", resp.status)


In [ ]:
def read_json(url: str):
    with urllib.request.urlopen(url) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))

catalog_status, catalog_body = read_json(f"{base_url}/management/v1/catalogs/PAIMON/{catalog_id}")
print("catalog fetch:", catalog_status)
print("mode:", catalog_body.get("mode"))


## Spark SQL operations through Paimon REST catalog

This section demonstrates create/insert/select/alter/view/delete/drop operations via Spark SQL.

In [ ]:
import uuid
from pyspark.sql import SparkSession

spark_catalog = "kasanari_paimon"

spark = (
    SparkSession.builder
    .appName("kasanari-paimon-proxy-ops")
    .config(
        "spark.jars.packages",
        "org.apache.paimon:paimon-spark-4.0_2.13:1.4.1,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "org.postgresql:postgresql:42.7.3",
    )
    .config(f"spark.sql.catalog.{spark_catalog}", "org.apache.paimon.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{spark_catalog}.metastore", "rest")
    .config(f"spark.sql.catalog.{spark_catalog}.uri", "http://kasanari:9090/paimon/v1")
    .config(f"spark.sql.catalog.{spark_catalog}.warehouse", catalog_id)
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

db = "demo"
table = f"events_{uuid.uuid4().hex[:8]}"
view = f"{table}_v"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {spark_catalog}.{db}")
spark.sql(
    f"""
    CREATE TABLE {spark_catalog}.{db}.{table} (
      id INT,
      event STRING,
      source STRING
    ) USING paimon
    TBLPROPERTIES ('bucket'='1')
    """
)

spark.sql(
    f"""
    INSERT INTO {spark_catalog}.{db}.{table}
    VALUES
      (1, 'signup', 'spark'),
      (2, 'click', 'spark'),
      (3, 'purchase', 'spark')
    """
)

print("Initial rows:")
spark.sql(f"SELECT * FROM {spark_catalog}.{db}.{table} ORDER BY id").show(truncate=False)

spark.sql(f"ALTER TABLE {spark_catalog}.{db}.{table} ADD COLUMNS (notes STRING)")
spark.sql(f"UPDATE {spark_catalog}.{db}.{table} SET notes = 'ok' WHERE id IN (1, 2)")

spark.sql(
    f"CREATE OR REPLACE VIEW {spark_catalog}.{db}.{view} AS "
    f"SELECT id, event FROM {spark_catalog}.{db}.{table} WHERE id <= 2"
)

print("View rows:")
spark.sql(f"SELECT * FROM {spark_catalog}.{db}.{view} ORDER BY id").show(truncate=False)

spark.sql(f"DELETE FROM {spark_catalog}.{db}.{table} WHERE id = 3")

print("After delete:")
spark.sql(f"SELECT id, event, notes FROM {spark_catalog}.{db}.{table} ORDER BY id").show(truncate=False)

spark.sql(f"DROP VIEW {spark_catalog}.{db}.{view}")
spark.sql(f"DROP TABLE {spark_catalog}.{db}.{table}")

print("Done: created, inserted, selected, altered, viewed, deleted, and dropped objects.")